# AI-GIS — End-to-End Pipeline (v0.5)
### Honeypot-Trained Hybrid Detection Model for Web Injection Attacks

This notebook runs the complete AI-GIS pipeline described in the methodology:

1. **Build / load** the labelled honeypot dataset (WEB-IDS23-seeded).
2. **Preprocess** each request and split by session (no leakage).
3. **Train** the three-part stacked model: Random Forest + LSTM + logistic-regression meta-learner.
4. **Evaluate** on a held-out adversarial benchmark.
5. **Compare** against a ModSecurity CRS PL2 baseline.
6. **Report** metrics, ablation, McNemar's test, and figures.

The code is self-contained — it does not import the project scripts — so it can run in
Google Colab or a local Jupyter environment on its own. A fixed random seed (42) is used
throughout for reproducibility.

> **Scope note (v0.5).** This iteration uses the initial honeypot-based dataset. The models,
> features, training, evaluation, and ModSecurity comparison are all real. Where a value cannot
> be established from the current run, it is left as a clearly marked placeholder.

## 0 · Setup and Reproducibility

In [ ]:
import os, sys, json, math, random, hashlib, urllib.parse
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# TensorFlow / Keras for the LSTM branch
import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow import keras
from tensorflow.keras import layers

# scikit-learn for the Random Forest, meta-learner, metrics, resampling
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, average_precision_score)

print("TensorFlow", tf.__version__)
print("Seed fixed at", SEED)

## 1 · Dataset

The dataset is a labelled honeypot log: each record is one HTTP request with a label
(1 = attack, 0 = benign), an attack family, and provenance flags. Attack payloads are produced
by family-specific generators and placed on WEB-IDS23 flow metadata.

**Point the path below at your honeypot log** (JSON-lines, one record per line). If the file is
not present, the next cell builds a small **illustrative sample** so the notebook still runs
end-to-end; replace it with the full corpus for real results.

In [ ]:
DATA_PATH = Path("data/honeypot_final.log")   # <-- set to your honeypot log

def load_log(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows

if DATA_PATH.exists():
    records = load_log(DATA_PATH)
    print(f"Loaded {len(records):,} records from {DATA_PATH}")
else:
    print("[!] Honeypot log not found — building a SMALL ILLUSTRATIVE SAMPLE.")
    print("    Replace DATA_PATH with the real corpus for actual results.")
    rng = random.Random(SEED)
    sqli = ["' OR '1'='1", "1 UNION SELECT NULL,NULL--", "'; DROP TABLE users--",
            "1 AND SLEEP(5)", "admin'--", "1' OR pwd=0x61--"]
    xss  = ["<script>alert(1)</script>", "<img src=x onerror=alert(1)>",
            "<svg/onload=alert(1)>", "\"><script>alert(document.cookie)</script>"]
    benign = ["/search?q=laptop review 2024", "if (a==b) { return true; }",
              "SELECT name FROM users WHERE id = %s  -- parameterised",
              "Re: invoice #4471, please review", "2026-01-01 INFO login ok"]
    records = []
    for i in range(1500):
        atk = rng.random() < 0.5
        if atk:
            fam = "sqli" if rng.random() < 0.6 else "xss"
            payload = rng.choice(sqli if fam == "sqli" else xss)
        else:
            fam, payload = "benign", rng.choice(benign)
        records.append({"uri": "/submit?q=" + urllib.parse.quote(payload),
                        "request_body": "", "method": "POST",
                        "session_id": f"s{i//3}", "label": int(atk),
                        "attack_family": fam})
    print(f"Built {len(records):,} sample records")

### 1.1 · Reconstruct the payload text and inspect composition
The attacker-controlled text is the URI query plus the request body.

In [ ]:
def payload_text(row):
    uri  = row.get("uri", "")
    body = row.get("request_body", "") or ""
    query = uri.split("?", 1)[1] if "?" in uri else ""
    # decode percent-encoding so the model sees the real characters
    text = f"{query} {body}".strip()
    try:
        text = urllib.parse.unquote(text)
    except Exception:
        pass
    return text

df = pd.DataFrame(records)
df["text"] = df.apply(payload_text, axis=1)
df["label"] = df["label"].astype(int)
df["session_id"] = df.get("session_id", pd.Series(range(len(df)))).astype(str)

print("Total records:", len(df))
print(df["label"].value_counts().rename({0: "benign", 1: "attack"}))
print("\nAttack families:")
print(df[df.label == 1]["attack_family"].value_counts())

## 2 · Preprocessing and Session-Grouped Split

The data is split by **session**, not by row, so no session appears in more than one partition.
This prevents leakage, because one session can produce several near-identical payloads.
The split is roughly 70% training, 15% validation, 15% testing.

In [ ]:
def normalize_text(t):
    return str(t).strip()

df["text"] = df["text"].map(normalize_text)

groups = df["session_id"].values
# 70% train, 30% temp
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
tr_idx, tmp_idx = next(gss1.split(df, groups=groups))
train_df = df.iloc[tr_idx].reset_index(drop=True)
tmp_df   = df.iloc[tmp_idx].reset_index(drop=True)
# split temp 50/50 -> 15% val, 15% test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
va_idx, te_idx = next(gss2.split(tmp_df, groups=tmp_df["session_id"].values))
val_df  = tmp_df.iloc[va_idx].reset_index(drop=True)
test_df = tmp_df.iloc[te_idx].reset_index(drop=True)

# leakage check: sessions must not cross splits
s_tr, s_va, s_te = set(train_df.session_id), set(val_df.session_id), set(test_df.session_id)
assert not (s_tr & s_te) and not (s_tr & s_va) and not (s_va & s_te), "session leakage!"
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("No session crosses a split boundary. [OK]")

## 3 · Feature Engineering (Random Forest branch)

The Random Forest reads a fixed-length numeric vector per request: simple aggregate counts,
structural attack-pattern flags, and character n-gram TF-IDF weights.

In [ ]:
SQLI_KEYWORDS = ["select", "union", "or ", "and ", "sleep", "drop", "--", "/*", "0x"]
XSS_KEYWORDS  = ["<script", "onerror", "onload", "javascript:", "<svg", "<img", "alert("]

def shannon_entropy(s):
    if not s:
        return 0.0
    from collections import Counter
    n = len(s)
    return -sum((c/n) * math.log2(c/n) for c in Counter(s).values())

def aggregate_features(text):
    lower = text.lower()
    return {
        "payload_len": len(text),
        "entropy": round(shannon_entropy(text), 4),
        "num_special_chars": sum(1 for c in text if not c.isalnum() and not c.isspace()),
        "num_digits": sum(c.isdigit() for c in text),
        "num_uppercase": sum(c.isupper() for c in text),
        "has_sqli_keyword": int(any(k in lower for k in SQLI_KEYWORDS)),
        "has_xss_keyword": int(any(k in lower for k in XSS_KEYWORDS)),
        "quote_count": text.count("'") + text.count('"'),
        "comment_token_count": text.count("--") + text.count("/*") + text.count("#"),
        "special_char_ratio": round(sum(1 for c in text if not c.isalnum() and not c.isspace()) / max(len(text), 1), 4),
    }

# character n-gram TF-IDF, fitted on TRAIN text only
NGRAM = 300
vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=NGRAM, lowercase=False)
vec.fit(train_df["text"])

def rf_matrix(texts):
    agg = pd.DataFrame([aggregate_features(t) for t in texts]).reset_index(drop=True)
    ng  = pd.DataFrame(vec.transform(texts).toarray(),
                       columns=[f"ngram_{i}" for i in range(NGRAM)]).reset_index(drop=True)
    return pd.concat([agg, ng], axis=1)

Xtr = rf_matrix(train_df["text"]); ytr = train_df["label"].to_numpy()
Xva = rf_matrix(val_df["text"]);   yva = val_df["label"].to_numpy()
Xte = rf_matrix(test_df["text"]);  yte = test_df["label"].to_numpy()
print("RF feature vector width:", Xtr.shape[1])

## 4 · Character Sequence (LSTM branch)

The LSTM reads the payload as an ordinal character sequence, padded or cut to 200 positions.
Padding is position 0, and the Embedding layer is told to **mask** it so empty positions are
ignored — without this the network does not learn.

In [ ]:
MAXLEN = 200

def ordinal_encode(text, max_len=MAXLEN):
    arr = np.zeros(max_len, dtype=np.int32)      # 0 = PAD
    for i, c in enumerate(text[:max_len]):
        code = ord(c)
        arr[i] = code if code <= 127 else 1      # 1 = UNK
    return arr

Etr = np.stack([ordinal_encode(t) for t in train_df["text"]])
Eva = np.stack([ordinal_encode(t) for t in val_df["text"]])
Ete = np.stack([ordinal_encode(t) for t in test_df["text"]])
print("LSTM input shape:", Etr.shape)

## 5 · Train the Three-Part Stacked Model

- **Random Forest** on the 319-value feature vector.
- **LSTM** on the character sequence, keeping the best checkpoint by validation AUC.
- **Logistic-regression meta-learner** fitted on the two branches' **validation** probabilities,
  so it never sees the branches' own training rows.

In [ ]:
# --- Random Forest branch ---
rf = RandomForestClassifier(n_estimators=200, max_depth=20, class_weight="balanced",
                            random_state=SEED, n_jobs=-1)
rf.fit(Xtr, ytr)
print("Random Forest trained.")

# --- LSTM branch ---
def build_lstm(seed=SEED):
    init = keras.initializers.GlorotUniform(seed=seed)
    m = keras.Sequential([
        layers.Input(shape=(MAXLEN,)),
        layers.Embedding(input_dim=128, output_dim=32, mask_zero=True),   # mask padding
        layers.LSTM(64, return_sequences=True, kernel_initializer=init),
        layers.Dropout(0.3, seed=seed),
        layers.LSTM(32, kernel_initializer=init),
        layers.Dropout(0.3, seed=seed),
        layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    return m

lstm = build_lstm()
cbs = [keras.callbacks.ModelCheckpoint("lstm_best.keras", monitor="val_auc", mode="max", save_best_only=True),
       keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)]
lstm.fit(Etr, ytr, validation_data=(Eva, yva), epochs=8, batch_size=128, callbacks=cbs, verbose=2)
print("LSTM trained.")

# --- Meta-learner on VALIDATION probabilities ---
rf_va   = rf.predict_proba(Xva)[:, 1]
lstm_va = lstm.predict(Eva, verbose=0).ravel()
meta = LogisticRegression(max_iter=1000)
meta.fit(np.column_stack([rf_va, lstm_va]), yva)
print("Meta-learner fitted on validation probabilities.")

## 6 · Prediction Helper

Combine the two branch probabilities through the meta-learner into one final score.

In [ ]:
def predict_proba(texts):
    X = rf_matrix(texts); rf_p = rf.predict_proba(X)[:, 1]
    E = np.stack([ordinal_encode(t) for t in texts]); ls_p = lstm.predict(E, verbose=0).ravel()
    return meta.predict_proba(np.column_stack([rf_p, ls_p]))[:, 1]

def metrics(y_true, scores, thr=0.5):
    pred = (scores >= thr).astype(int)
    out = {"n": len(y_true),
           "recall": recall_score(y_true, pred, zero_division=0),
           "precision": precision_score(y_true, pred, zero_division=0),
           "f1": f1_score(y_true, pred, zero_division=0)}
    neg = (y_true == 0)
    out["fpr"] = float(pred[neg].mean()) if neg.any() else float("nan")
    try:
        out["auc"] = roc_auc_score(y_true, scores)
    except ValueError:
        out["auc"] = float("nan")
    return out

## 7 · Held-Out Evaluation Benchmark

Point this at your held-out benchmark CSV (`text,label` columns, kept entirely out of training).
If none is supplied, the internal **test split** from Section 2 is used as a fallback so the
notebook runs end-to-end.

> Replace the fallback with the frozen held-out benchmark for the reported results.

In [ ]:
BENCHMARK_CSV = Path("data/eval/holdout_benchmark.csv")   # <-- set to your held-out benchmark

if BENCHMARK_CSV.exists():
    bench = pd.read_csv(BENCHMARK_CSV)
    bench["label"] = bench["label"].astype(int)
    print(f"Loaded held-out benchmark: {len(bench)} records")
else:
    print("[!] Benchmark CSV not found — using the internal TEST split as a fallback.")
    bench = test_df[["text", "label"]].copy()

bench_scores = predict_proba(bench["text"].tolist())
overall = metrics(bench["label"].to_numpy(), bench_scores)
print("\nAI-GIS on the benchmark:")
for k, v in overall.items():
    print(f"  {k:10s} {v:.4f}" if isinstance(v, float) else f"  {k:10s} {v}")

### 7.1 · Results by attack type
If the benchmark carries an `attack_type` (or `attack_family`) column, results are broken out
by SQLi and XSS.

In [ ]:
type_col = "attack_type" if "attack_type" in bench.columns else ("attack_family" if "attack_family" in bench.columns else None)
rows = [{"subset": "Combined", **overall}]
if type_col:
    for atype in ["sqli", "xss"]:
        mask = (bench["label"] == 1) & (bench[type_col].astype(str).str.lower().str.contains(atype))
        neg  = bench["label"] == 0
        idx = mask | neg
        if mask.sum() > 0:
            m = metrics(bench["label"].to_numpy()[idx.values], bench_scores[idx.values])
            rows.append({"subset": atype.upper(), **m})
results_df = pd.DataFrame(rows)[["subset", "n", "recall", "precision", "f1", "fpr", "auc"]]
results_df.round(4)

## 8 · ModSecurity CRS PL2 Baseline (optional — needs Docker)

ModSecurity is the rule-based baseline. It runs as a Docker container and inspects the **same**
records independently — it does not exchange data with AI-GIS. Each payload is sent as a GET query
and a POST body; a record counts as detected if either is blocked (HTTP 403).

This cell is skipped automatically if the ModSecurity endpoint is not reachable (e.g. on Colab).

In [ ]:
import urllib.request, urllib.error

MODSEC_URL = "http://localhost:8080"   # owasp/modsecurity-crs:nginx, Paranoia Level 2
BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/124 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8",
}

def modsec_available():
    try:
        urllib.request.urlopen(MODSEC_URL + "/get?q=test", timeout=5)
        return True
    except urllib.error.HTTPError:
        return True          # a 4xx still means the server answered
    except Exception:
        return False

def modsec_blocked(payload):
    for method in ("get", "post"):
        try:
            if method == "get":
                req = urllib.request.Request(MODSEC_URL + "/get?q=" + urllib.parse.quote(payload, safe=""),
                                             headers=BROWSER_HEADERS)
            else:
                req = urllib.request.Request(MODSEC_URL + "/post",
                                             data=urllib.parse.urlencode({"q": payload}).encode(),
                                             headers={**BROWSER_HEADERS, "Content-Type": "application/x-www-form-urlencoded"})
            urllib.request.urlopen(req, timeout=15)
        except urllib.error.HTTPError as e:
            if e.code == 403:
                return 1
        except Exception:
            pass
    return 0

if modsec_available():
    print("ModSecurity reachable — scoring the benchmark (this may take a few minutes)...")
    ms_pred = np.array([modsec_blocked(p) for p in bench["text"].tolist()])
    yb = bench["label"].to_numpy()
    ms = {"recall": recall_score(yb, ms_pred, zero_division=0),
          "precision": precision_score(yb, ms_pred, zero_division=0),
          "f1": f1_score(yb, ms_pred, zero_division=0),
          "fpr": float(ms_pred[yb == 0].mean()),
          "accuracy": float((ms_pred == yb).mean())}
    print("ModSecurity CRS PL2:", {k: round(v, 4) for k, v in ms.items()})
else:
    print("[skip] ModSecurity endpoint not reachable — baseline comparison skipped.")
    ms_pred, ms = None, None

### 8.1 · AI-GIS vs ModSecurity + McNemar's paired test
McNemar's exact test checks whether the two systems' disagreements on the same records are
statistically meaningful.

In [ ]:
if ms_pred is not None:
    yb = bench["label"].to_numpy()
    ai_pred = (bench_scores >= 0.5).astype(int)
    ai_ok, ms_ok = (ai_pred == yb), (ms_pred == yb)
    b = int((ai_ok & ~ms_ok).sum())    # AI right, ModSec wrong
    c = int((~ai_ok & ms_ok).sum())    # AI wrong, ModSec right
    from scipy.stats import binomtest
    p_value = binomtest(min(b, c), n=b + c, p=0.5, alternative="two-sided").pvalue if (b + c) else 1.0

    comp = pd.DataFrame({
        "Metric": ["Recall", "Precision", "F1", "FPR", "Accuracy"],
        "AI-GIS": [overall["recall"], overall["precision"], overall["f1"], overall["fpr"], float(ai_ok.mean())],
        "ModSecurity PL2": [ms["recall"], ms["precision"], ms["f1"], ms["fpr"], ms["accuracy"]],
    })
    display(comp.round(4))
    print(f"\nMcNemar: AI right/ModSec wrong = {b}, AI wrong/ModSec right = {c}, p = {p_value:.2e}")
else:
    print("Baseline comparison unavailable (ModSecurity was skipped).")

## 9 · Ablation — Does Stacking Help?

Compare the full stack against its parts on the same benchmark, to test whether combining the
branches adds value over the best single model.

In [ ]:
def branch_scores(texts):
    X = rf_matrix(texts); rf_p = rf.predict_proba(X)[:, 1]
    E = np.stack([ordinal_encode(t) for t in texts]); ls_p = lstm.predict(E, verbose=0).ravel()
    return rf_p, ls_p

yb = bench["label"].to_numpy()
rf_b, ls_b = branch_scores(bench["text"].tolist())
stack_b = meta.predict_proba(np.column_stack([rf_b, ls_b]))[:, 1]

abl = pd.DataFrame([
    {"condition": "Random Forest alone", **metrics(yb, rf_b)},
    {"condition": "LSTM alone",          **metrics(yb, ls_b)},
    {"condition": "Full stack (RF+LSTM+meta)", **metrics(yb, stack_b)},
])[["condition", "f1", "precision", "recall", "fpr", "auc"]]
display(abl.round(4))

best_single = max(metrics(yb, rf_b)["f1"], metrics(yb, ls_b)["f1"])
print(f"\nBest single-branch F1 = {best_single:.4f}  |  Full stack F1 = {metrics(yb, stack_b)['f1']:.4f}")
print("Stacking adds value." if metrics(yb, stack_b)["f1"] > best_single
      else "Stacking does NOT beat the best single branch on this benchmark.")

## 10 · Figures

ROC curve for AI-GIS, with the ModSecurity operating point overlaid when available.

In [ ]:
import matplotlib.pyplot as plt

fpr_c, tpr_c, _ = roc_curve(yb, bench_scores)
plt.figure(figsize=(6.5, 5.5))
plt.plot(fpr_c, tpr_c, color="#059669", lw=2, label=f"AI-GIS (AUC = {overall['auc']:.3f})")
if ms_pred is not None:
    plt.scatter([ms["fpr"]], [ms["recall"]], color="#c0392b", s=120, marker="X",
                zorder=5, label=f"ModSecurity PL2 (FPR={ms['fpr']:.2f})")
plt.plot([0, 1], [0, 1], "--", color="#888", lw=1)
plt.xlabel("False-Positive Rate"); plt.ylabel("True-Positive Rate (Recall)")
plt.title("ROC Curve: AI-GIS vs ModSecurity CRS PL2")
plt.legend(loc="lower right"); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()

## 11 · Save Results

Write the metrics tables to CSV / Excel for the results chapter.

In [ ]:
out_dir = Path("reports"); out_dir.mkdir(exist_ok=True)
results_df.to_csv(out_dir / "aigis_results_by_type.csv", index=False)
try:
    with pd.ExcelWriter(out_dir / "aigis_results.xlsx") as xw:
        results_df.to_excel(xw, sheet_name="by_type", index=False)
        abl.to_excel(xw, sheet_name="ablation", index=False)
        if ms_pred is not None:
            comp.to_excel(xw, sheet_name="vs_modsecurity", index=False)
    print("Saved reports/aigis_results.xlsx")
except Exception as e:
    print("Excel save skipped:", e)
print("Saved reports/aigis_results_by_type.csv")

---
### Notes on reproducibility and scope
- All randomness is seeded (`SEED = 42`); the Random Forest and split reproduce exactly, and the
  LSTM reproduces to numerical tolerance.
- This notebook uses the **initial honeypot-based dataset**. Swap in the full corpus and the frozen
  held-out benchmark for the final reported numbers.
- The models, features, training, evaluation, ablation, and ModSecurity comparison mirror the
  methodology chapter one-to-one.